<a href="https://colab.research.google.com/github/johnjoseph004/AI-ML-internship/blob/main/day7_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
df = pd.read_csv("/content/IMDB Dataset.csv", encoding ="latin1", on_bad_lines='skip')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [4]:
import re
negation_words = [
    "not good", "not bad", "not great", "don't like "
    "didn't like", "never liked", "wasn't good",
    "isn't good", "no good"
]

def clean_text(text):
  text = text.lower()

  text = re.sub(r"[^a-zA-Z\s']"," ", text)

  for phrase in negation_words:
    text = text.replace(phrase, phrase.replace(" ", "_")) # Changed to underscore for clarity

  return text

In [5]:
df['review'] = df['review'].apply(clean_text)

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['review'],df['sentiment'],test_size=0.2,random_state=42)

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq,maxlen=max_len,padding='post')
X_test_pad = pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dropout,Dense

model = Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128, dropout = 0.3,recurrent_dropout = 0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')

])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(X_train_pad,y_train,epochs=5,batch_size=64,validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 237s 473ms/step - accuracy: 0.5919 - loss: 0.6269 - val_accuracy: 0.5960 - val_loss: 0.6344
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 256s 462ms/step - accuracy: 0.6957 - loss: 0.5278 - val_accuracy: 0.8432 - val_loss: 0.3963
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 257s 452ms/step - accuracy: 0.8675 - loss: 0.3392 - val_accuracy: 0.8339 - val_loss: 0.4160
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 242s 484ms/step - accuracy: 0.9077 - loss: 0.2539 - val_accuracy: 0.8669 - val_loss: 0.3828
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 260s 480ms/step - accuracy: 0.9382 - loss: 0.1851 - val_accuracy: 0.8645 - val_loss: 0.3805


In [12]:
loss,acc = model.evaluate(X_test_pad,y_test)

print("Test Accuracy:",acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 20s 62ms/step - accuracy: 0.8688 - loss: 0.3550
Test Accuracy: 0.8687999844551086


In [13]:
def predict_sentiment(review):
  review = clean_text(review)
  seq = tokenizer.texts_to_sequences([review])
  padded = pad_sequences(seq,maxlen=max_len,padding='post')
  prediction = model.predict(padded)[0][0]

  print("\nReview:",review)
  print("Score:",prediction)

  if prediction >= 0.5:
    print("Sentiment: Positive 😍")
  else:
    print("Sentiment: Negative 🤦‍♂️")


In [16]:
predict_sentiment("this movie is amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step

Review: this movie is amazing
Score: 0.86596686
Sentiment: Positive 😍
